# Integración: Modelo Tabular + ResNet50

Combinamos las predicciones del modelo tabular (LightGBM) con las del modelo de imágenes (ResNet50) mediante **blending**: sumamos los scores crudos de ambos modelos y tomamos el argmax.

**Flujo:**
1. Cargar datos y hacer el mismo split que usó ResNet
2. Entrenar LightGBM sobre datos tabulares
3. Cargar predicciones del ResNet
4. Blend: sumar scores de ambos modelos
5. Comparar: LGB solo vs ResNet solo vs Blended

## 1. Imports y configuración

In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score, confusion_matrix
from joblib import load

BASE_DIR       = '../'
PATH_TRAIN_CSV = os.path.join(BASE_DIR, 'input/petfinder-adoption-prediction/train/train.csv')
PATH_MODELS    = os.path.join(BASE_DIR, 'work/models')

# El mismo seed y split que usó ResNet — imprescindible para que los val sets coincidan
SEED      = 42
TEST_SIZE = 0.2

# Archivo de predicciones del mejor ResNet (Kappa 0.3194)
RESNET_PREDS_FILE = os.path.join(PATH_MODELS, 'resnet50_val_preds_20260429_203351.joblib')

print('Configuración lista.')

## 2. Cargar datos y split

In [ ]:
df = pd.read_csv(PATH_TRAIN_CSV)

# MISMO split que usó ResNet para que los PetIDs de val coincidan
train_data, val_data = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=df['AdoptionSpeed']
)

print(f'Train: {len(train_data)} | Val: {len(val_data)}')

## 3. Modelo tabular: LightGBM

In [ ]:
features = [
    'Type', 'Age', 'Breed1', 'Breed2', 'Gender',
    'Color1', 'Color2', 'Color3', 'MaturitySize', 'FurLength',
    'Vaccinated', 'Dewormed', 'Sterilized', 'Health',
    'Quantity', 'Fee', 'State', 'VideoAmt', 'PhotoAmt'
]
label = 'AdoptionSpeed'

X_train = train_data[features]
y_train = train_data[label]
X_val   = val_data[features]
y_val   = val_data[label]

lgb_params = {
    'objective':       'multiclass',
    'num_class':       5,
    'verbosity':       -1,
    'num_leaves':      156,
    'feature_fraction': 0.473,
    'bagging_fraction': 0.450,
    'bagging_freq':    2,
    'min_child_samples': 86,
    'lambda_l1':       5e-8,
    'lambda_l2':       4e-4,
}

lgb_train_dataset = lgb.Dataset(data=X_train, label=y_train)
lgb_model = lgb.train(lgb_params, lgb_train_dataset)

# Predicciones sobre val (probabilidades por clase)
lgb_scores = lgb_model.predict(X_val)  # shape (n, 5)
lgb_preds  = lgb_scores.argmax(axis=1)

kappa_lgb = cohen_kappa_score(y_val, lgb_preds, weights='quadratic')
print(f'Kappa LightGBM (tabular): {kappa_lgb:.4f}')

## 4. Cargar predicciones del ResNet

In [ ]:
resnet_df = load(RESNET_PREDS_FILE)
print(f'Predicciones ResNet cargadas: {len(resnet_df)} registros')
resnet_df.head(3)

## 5. Merge y Blend

Unimos ambas predicciones por `PetID` y sumamos los scores crudos.
El argmax de la suma es la predicción final del modelo blended.

In [ ]:
# Armamos el df de LGB con PetID
lgb_df = val_data[['PetID', label]].copy()
lgb_df['lgb_score'] = [lgb_scores[i] for i in range(len(lgb_scores))]
lgb_df['lgb_pred']  = lgb_preds

# Merge con ResNet por PetID
merged = lgb_df.merge(
    resnet_df[['PetID', 'resnet_pred_score', 'resnet_pred']],
    on='PetID',
    how='inner'
)

print(f'Registros en común: {len(merged)} de {len(val_data)} val')

# Blend: suma de scores crudos
merged['blend_score'] = [
    np.array(r['lgb_score']) + np.array(r['resnet_pred_score'])
    for _, r in merged.iterrows()
]
merged['blend_pred'] = [s.argmax() for s in merged['blend_score']]

merged.head(3)

## 6. Comparación de modelos

In [ ]:
y_true      = merged[label]
lgb_pred    = merged['lgb_pred']
resnet_pred = merged['resnet_pred']
blend_pred  = merged['blend_pred']

kappa_lgb    = cohen_kappa_score(y_true, lgb_pred,    weights='quadratic')
kappa_resnet = cohen_kappa_score(y_true, resnet_pred, weights='quadratic')
kappa_blend  = cohen_kappa_score(y_true, blend_pred,  weights='quadratic')

resultados = pd.DataFrame({
    'Modelo':  ['LightGBM (tabular)', 'ResNet50 (imágenes)', 'Blended (LGB + ResNet)'],
    'Kappa':   [kappa_lgb, kappa_resnet, kappa_blend]
})
print(resultados.to_string(index=False))

# Gráfico comparativo
plt.figure(figsize=(8, 4))
colors = ['steelblue', 'salmon', 'seagreen']
plt.bar(resultados['Modelo'], resultados['Kappa'], color=colors)
plt.ylabel('Kappa cuadrático')
plt.title('Comparación de modelos')
plt.ylim(0, 0.5)
for i, v in enumerate(resultados['Kappa']):
    plt.text(i, v + 0.005, f'{v:.4f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Matrices de confusión de los tres modelos
CLASS_NAMES = ['0', '1', '2', '3', '4']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, preds, titulo in zip(axes,
                              [lgb_pred, resnet_pred, blend_pred],
                              [f'LightGBM  Kappa: {kappa_lgb:.4f}',
                               f'ResNet50  Kappa: {kappa_resnet:.4f}',
                               f'Blended   Kappa: {kappa_blend:.4f}']):
    cm = confusion_matrix(y_true, preds)
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
    sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(titulo)
    ax.set_xlabel('Predicho')
    ax.set_ylabel('Real')

plt.tight_layout()
plt.show()

## 7. Blend con peso ajustable

En lugar de sumar con igual peso, probamos darle más o menos importancia a cada modelo.

In [ ]:
resultados_pesos = []

for w_lgb in np.arange(0.0, 1.1, 0.1):
    w_resnet = 1.0 - w_lgb
    blend_w = [
        w_lgb * np.array(r['lgb_score']) + w_resnet * np.array(r['resnet_pred_score'])
        for _, r in merged.iterrows()
    ]
    preds_w = [s.argmax() for s in blend_w]
    kappa_w = cohen_kappa_score(y_true, preds_w, weights='quadratic')
    resultados_pesos.append({'w_lgb': round(w_lgb, 1), 'w_resnet': round(w_resnet, 1), 'kappa': kappa_w})

df_pesos = pd.DataFrame(resultados_pesos)

plt.figure(figsize=(10, 4))
plt.plot(df_pesos['w_lgb'], df_pesos['kappa'], marker='o', color='seagreen')
plt.xlabel('Peso LightGBM (peso ResNet = 1 - peso LGB)')
plt.ylabel('Kappa')
plt.title('Kappa según peso del blend')
plt.xticks(df_pesos['w_lgb'])
plt.tight_layout()
plt.show()

mejor = df_pesos.loc[df_pesos['kappa'].idxmax()]
print(f'\nMejor combinación: w_lgb={mejor.w_lgb} / w_resnet={mejor.w_resnet} → Kappa: {mejor.kappa:.4f}')
print(df_pesos.to_string(index=False))